# BTC Kalshi Time-Series Model

This notebook predicts the BTC Kalshi YES contract midpoint `n` seconds in the future. It follows the same modeling flow as `log_reg.ipynb`, but treats the question as a regression/time-series problem instead of a binary classification problem.

The target is created from current data as `yes_mid_dollars_t_plus_{n}s`. Future columns such as `next_price_dollars_lead1`, `outcome`, and any generated `t_plus` columns are excluded from the features to avoid leakage.

In [8]:
from pathlib import Path
import sys
import json
import pickle
import numpy as np
import polars as pl

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / "kalshi_timeseries_model.py").exists() else Path.cwd() / "models"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from kalshi_timeseries_model import (
    DEFAULT_DATA_PATH,
    DEFAULT_OUTPUT_DIR,
    add_future_target,
    add_past_only_features,
    build_model,
    chronological_event_split,
    evaluate,
    make_xy,
    parse_args,
    read_final_data,
    resolve_data_path,
)

class NotebookArgs:
    data_path = DEFAULT_DATA_PATH
    output_dir = DEFAULT_OUTPUT_DIR
    horizon_seconds = 30
    test_size = 0.2
    max_iter = 300
    learning_rate = 0.05
    max_leaf_nodes = 31
    model_type = "auto"
    ridge_alpha = 10.0

args = NotebookArgs()
data_path = resolve_data_path(args.data_path)
target_col = f"yes_mid_dollars_t_plus_{args.horizon_seconds}s"
args.output_dir.mkdir(parents=True, exist_ok=True)

target_col

'yes_mid_dollars_t_plus_30s'

## Load Data

The preferred path is `data_gather/final_data/final_data.csv`. If that file is not present, the model falls back to `final_data/final_data.csv`.

In [9]:
df = read_final_data(data_path)
df.shape, df.columns[:10]

((1017826, 54),
 ['curr_time',
  'open_time',
  'close_time',
  'prev_time',
  'time_to_close',
  'last_price_dollars',
  'yes_mid_dollars',
  'yes_spread_dollars',
  'distance_from_strike',
  'yes_mid_change_1s'])

## Build Future Target And Past-Only Features

The target is joined by timestamp: current `curr_time + n seconds` to the next available future quote within the same Kalshi event. Added lag and rolling features only use past rows inside the event.

In [3]:
df = add_future_target(df, args.horizon_seconds)
df = add_past_only_features(df)

df.select(["curr_time", "open_time", "close_time", "yes_mid_dollars", target_col]).head()

curr_time,open_time,close_time,yes_mid_dollars,yes_mid_dollars_t_plus_30s
"datetime[μs, UTC]","datetime[μs, UTC]","datetime[μs, UTC]",f64,f64
2026-03-21 10:47:33.863137 UTC,2026-03-21 10:45:00 UTC,2026-03-21 11:00:00 UTC,0.41,0.43
2026-03-21 10:47:34.863913 UTC,2026-03-21 10:45:00 UTC,2026-03-21 11:00:00 UTC,0.41,0.43
2026-03-21 10:47:35.864626 UTC,2026-03-21 10:45:00 UTC,2026-03-21 11:00:00 UTC,0.41,0.43
2026-03-21 10:47:36.866221 UTC,2026-03-21 10:45:00 UTC,2026-03-21 11:00:00 UTC,0.41,0.43
2026-03-21 10:47:37.866620 UTC,2026-03-21 10:45:00 UTC,2026-03-21 11:00:00 UTC,0.41,0.43


## Chronological Event Split

Rows are split by Kalshi event time, so the test set contains later 15-minute markets rather than randomly mixed rows from the same event.

In [4]:
train_df, test_df = chronological_event_split(df, args.test_size)
train_df.shape, test_df.shape

((778703, 63), (194996, 63))

## Feature Matrix

`make_xy` drops known leakage columns: `outcome`, `next_price_dollars_lead1`, any `next_*`, any column containing `lead`, and all generated `t_plus` target columns.

In [5]:
x_train, y_train, feature_cols = make_xy(train_df, target_col)
x_test, y_test, _ = make_xy(test_df, target_col, feature_cols)

len(feature_cols), feature_cols[:20], x_train.shape, x_test.shape

(56,
 ['time_to_close',
  'last_price_dollars',
  'yes_mid_dollars',
  'yes_spread_dollars',
  'distance_from_strike',
  'yes_mid_change_1s',
  'yes_mid_change_5s',
  'yes_mid_change_std_30s',
  'yes_mid_change_std_60s',
  'yes_spread_mean_30s',
  'ETH_last_price_dollars',
  'ETH_yes_mid_dollars',
  'ETH_yes_spread_dollars',
  'ETH_distance_from_strike',
  'ETH_yes_mid_change_1s',
  'ETH_yes_mid_change_5s',
  'ETH_yes_mid_change_std_30s',
  'ETH_yes_mid_change_std_60s',
  'ETH_yes_spread_mean_30s',
  'XRP_last_price_dollars'],
 (721883, 56),
 (180776, 56))

## Train Model

When `scikit-learn` is installed, `auto` uses histogram gradient boosting. Otherwise it falls back to a dependency-free ridge ARX model.

In [6]:
model = build_model(args, n_features=len(feature_cols))
model.fit(x_train, y_train)

type(model).__name__

'RidgeARXRegressor'

## Evaluate And Save Artifacts

In [7]:
predictions = model.predict(x_test)
metrics = evaluate(y_test, predictions)

metrics_payload = {
    "target": target_col,
    "horizon_seconds": args.horizon_seconds,
    "data_path": str(data_path),
    "n_rows": df.height,
    "n_train_rows": int(len(y_train)),
    "n_test_rows": int(len(y_test)),
    "n_features": len(feature_cols),
    "model_type": type(model).__name__,
    "features": feature_cols,
    "metrics": metrics,
}

metrics_path = args.output_dir / f"metrics_{args.horizon_seconds}s.json"
model_path = args.output_dir / f"model_{args.horizon_seconds}s.pkl"
predictions_path = args.output_dir / f"predictions_{args.horizon_seconds}s.csv"

metrics_path.write_text(json.dumps(metrics_payload, indent=2))
with model_path.open("wb") as model_file:
    pickle.dump({
        "model": model,
        "feature_cols": feature_cols,
        "target_col": target_col,
        "horizon_seconds": args.horizon_seconds,
    }, model_file)

pl.DataFrame({
    "actual": y_test,
    "predicted": predictions,
    "error": predictions - y_test,
}).write_csv(predictions_path)

metrics_payload

{'target': 'yes_mid_dollars_t_plus_30s',
 'horizon_seconds': 30,
 'data_path': '/Users/tahmidwashy/mlproj/crypto_pred_market/final_data/final_data.csv',
 'n_rows': 973699,
 'n_train_rows': 721883,
 'n_test_rows': 180776,
 'n_features': 56,
 'model_type': 'RidgeARXRegressor',
 'features': ['time_to_close',
  'last_price_dollars',
  'yes_mid_dollars',
  'yes_spread_dollars',
  'distance_from_strike',
  'yes_mid_change_1s',
  'yes_mid_change_5s',
  'yes_mid_change_std_30s',
  'yes_mid_change_std_60s',
  'yes_spread_mean_30s',
  'ETH_last_price_dollars',
  'ETH_yes_mid_dollars',
  'ETH_yes_spread_dollars',
  'ETH_distance_from_strike',
  'ETH_yes_mid_change_1s',
  'ETH_yes_mid_change_5s',
  'ETH_yes_mid_change_std_30s',
  'ETH_yes_mid_change_std_60s',
  'ETH_yes_spread_mean_30s',
  'XRP_last_price_dollars',
  'XRP_yes_mid_dollars',
  'XRP_yes_spread_dollars',
  'XRP_distance_from_strike',
  'XRP_yes_mid_change_1s',
  'XRP_yes_mid_change_5s',
  'XRP_yes_mid_change_std_30s',
  'XRP_yes_mid_c